# 03 · Evaluation (FA / CER / F1)
Giai đoạn 3: đánh giá adapter (base **Qwen3-VL-8B** local) trên tập **Test** (chưa từng thấy lúc train).

**Báo cáo lưu vào Google Drive.**

## 1. Mount Drive + đường dẫn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_DRIVE = Path('/content/drive/MyDrive/cccd_project')
TEST_JSONL  = PROJECT_DRIVE / 'data' / 'dataset' / 'test.jsonl'
CKPT_DIR    = PROJECT_DRIVE / 'checkpoints' / 'qwen3vl-cccd-lora'
QWEN3_LOCAL = PROJECT_DRIVE / 'models' / 'Qwen3-VL-8B-Instruct'
REPORT_PATH = PROJECT_DRIVE / 'result' / 'eval_report.json'
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
print('Test  :', TEST_JSONL)
print('Base  :', QWEN3_LOCAL)
print('Report:', REPORT_PATH)

## 2. Cài thư viện (transformers từ source cho Qwen3-VL)

In [ ]:
!cp -r '/content/drive/MyDrive/cccd_project/code' /content/cccd 2>/dev/null || echo 'Đặt source vào /content/cccd'
%cd /content/cccd
!pip -q install git+https://github.com/huggingface/transformers
!pip -q install qwen-vl-utils accelerate peft bitsandbytes Pillow tqdm

## 3. Chạy đánh giá (base LOCAL + adapter)

In [ ]:
!python scripts/evaluate.py \
    --test_jsonl '{TEST_JSONL}' \
    --base_model '{QWEN3_LOCAL}' \
    --adapter_dir '{CKPT_DIR}' \
    --image_root '{PROJECT_DRIVE}' \
    --report_path '{REPORT_PATH}'

## 4. Xem báo cáo + per-field accuracy

In [ ]:
import json
with open(REPORT_PATH, encoding='utf-8') as f:
    report = json.load(f)
print(f"Field Accuracy : {report['field_accuracy']:.2%}")
print(f"CER            : {report['cer']:.4f}")
print(f"F1 (micro)     : {report['f1']:.2%}")
print('\nPer-field accuracy:')
for k, v in sorted(report['per_field_accuracy'].items(), key=lambda x: x[1]):
    print(f'  {k:20s} {v:.2%}')